# **Hito 1 · Preprocesado de documentos y consultas con spaCy**

---
**Proyecto PLN (UC3M, 2026/27)** — Chatbot RAG sobre documentación de misiones espaciales.

**Issue:** *Preprocesar documentos y consultas con spaCy*

* Aplicar **lematización**, **filtrado por categoría gramatical**, **eliminación de stopwords** y **filtrado por frecuencia documental**.
* Aplicar a las **consultas** exactamente el **mismo preprocesado** que a los documentos.

---

**Entrada:** `fragmentos_base/misiones_300.jsonl`, generado por `scripts/segmentar_corpus.py`. Los documentos originales de hasta **400 palabras** se conservan completos en un único fragmento; los de más de 400 se dividen con objetivo de **300 palabras**, tamaños de **200 a 400** y **45 palabras de solapamiento** (15 % del tamaño objetivo). Los documentos sin palabras reconocibles se omiten y contabilizan. La unidad de indexación es el **fragmento**, así que todas las estadísticas (frecuencia documental, etc.) se calculan sobre fragmentos, tal y como indica el enunciado del Hito 1.

**Salida** (carpeta `preprocesado/`), que reutilizarán los baselines TF-IDF y BM25:

| Fichero | Contenido |
|---|---|
| `lemas_spacy.jsonl` y `lemas_spacy.meta.json` | Lemas previos a n-gramas y frecuencia, con metadatos y huellas para validar su reutilización |
| `fragmentos_preprocesados.jsonl` | `chunk_id`, `doc_id` y `tokens` finales de cada fragmento |
| `diccionario.dict` | Diccionario de Gensim ya filtrado por frecuencia documental |
| `ngramas.phrases` | Modelo de n-gramas (Gensim `Phrases`, congelado) |
| `config_preprocesado.json` | Configuración, estadísticas, procedencia y manifiesto de integridad de la ejecución |
| `consultas_preprocesadas.jsonl` | Solo si existe `queries.jsonl` en la raíz del repositorio |

**Base:** el pipeline sigue el notebook *Text Vectorization I* (secciones 1.2, 1.3, 2.2, 2.3 y 4.1) y el *spaCy tutorial*. Las adaptaciones al corpus están marcadas con 🆕.

**Estado de esta revisión:** el segmentador está validado para 18.992 fragmentos de 3.314 documentos, pero la entrada y los artefactos guardados todavía corresponden a la colección anterior. Las salidas del notebook se han retirado para no confundirlas con resultados nuevos. La regeneración y la ejecución completa quedan para la segunda fase; los recuentos de las celdas se calcularán sobre la entrada que realmente se cargue.

## **0. Configuración**

In [ ]:
# Colab-specific helpers (they are skipped when running the notebook elsewhere)
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    # Fancy (interactive) display of dataframes
    get_ipython().run_line_magic('load_ext', 'google.colab.data_table')

# To wrap long text lines in outputs
from IPython.display import HTML, display
def set_css(*args, **kwargs):
    display(HTML('<style> pre { white-space: pre-wrap; } </style>'))
get_ipython().events.register('pre_run_cell', set_css)

In [ ]:
# Packages that are not preinstalled in Google Colab (en local: pip install -r requirements.txt)
%pip install --quiet gensim wordninja

import spacy
try:
    spacy.load('en_core_web_sm')
except OSError:
    !{sys.executable} -m spacy download en_core_web_sm

In [ ]:
import os
import json
import re
import time
import hashlib
from tempfile import TemporaryDirectory
from importlib.metadata import version as package_version
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import spacy
import wordninja
from gensim.corpora import Dictionary
from gensim.models.phrases import Phrases, FrozenPhrases

%matplotlib inline
plt.rcParams['figure.figsize'] = [8, 4]
pd.set_option('display.max_colwidth', 120)

print('spaCy', spacy.__version__)

### Ficheros de datos

El notebook está en la carpeta `notebooks/` del repositorio y lee los fragmentos de `fragmentos_base/misiones_300.jsonl`.

* **En local**, ejecutadlo desde `notebooks/`: `REPO_DIR = '..'` apunta a la raíz del repositorio.
* **En Google Colab**, copiad el repositorio a una carpeta de vuestro Google Drive (por ejemplo, con `!git clone` dentro de `/content/drive/My Drive/`) y actualizad `REPO_DIR` en consecuencia.

Los ficheros que genera este notebook se guardan en la carpeta `preprocesado/` del repositorio; los necesitaréis en el notebook de los baselines (TF-IDF y BM25).

In [ ]:
REPO_DIR = '..'   # raíz del repositorio cuando el notebook se ejecuta en local desde notebooks/

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO_DIR = '/content/drive/My Drive/uc3m-pln-rag-2026'  # UPDATE THIS ACCORDING TO WHERE YOU STORED THE REPOSITORY!!!!

print(os.listdir(REPO_DIR))

REPO_DIR = Path(REPO_DIR)
FRAGMENTS_PATH = REPO_DIR / 'fragmentos_base' / 'misiones_300.jsonl'
QUERIES_PATH = REPO_DIR / 'queries.jsonl'
OUT_DIR = REPO_DIR / 'preprocesado'
OUT_DIR.mkdir(exist_ok=True)

LEMMAS_CACHE = OUT_DIR / 'lemas_spacy.jsonl'
LEMMAS_META = OUT_DIR / 'lemas_spacy.meta.json'
FORCE_SPACY = False   # True para recalcular incluso si la caché es compatible

# Incrementar si cambia fix_spaced_letters, wrangle_text, normalize_token o lemmatize_texts.
# Es explícita porque inspect.getsource no es fiable entre kernels de Jupyter y Colab.
PREPROCESSING_VERSION = 1


Todos los parámetros del preprocesado están juntos aquí, para que sea fácil documentarlos en `MILESTONE_1.md` y probar variantes. Cada uno se justifica más abajo, en la sección donde se usa.

In [ ]:
CONFIG = {
    # spaCy
    'spacy_model': 'en_core_web_sm',
    'valid_pos': ['NOUN', 'VERB', 'ADJ', 'PROPN'],   # igual que en Text Vectorization I
    'min_token_len': 2,            # 🆕 descarta letras sueltas (restos de OCR, unidades 'm', 's'...)
    'keep_identifiers': True,      # 🆕 conserva números y códigos alfanuméricos (Apollo 11, STS-129, Astra 2B, 1969)
    'max_identifier_len': 4,
    'ocr_resegment': True,         # 🆕 re-segmenta con wordninja el texto OCR con letras separadas
    'specific_stopwords': [        # stopwords del dominio (en IMDB eran 'movie' y 'film')
        'et', 'al', 'fig', 'figure', 'table', 'page', 'ref', 'eq', 'etc',
    ],
    # N-gramas (Gensim Phrases), como en la sección 2.2 de Text Vectorization I
    'use_ngrams': True,
    'ngram_min_count': 5,
    'ngram_threshold': 20,
    # Filtro por frecuencia documental (Dictionary.filter_extremes), sección 2.3
    'no_below': 2,                 # nº mínimo de fragmentos en los que debe aparecer un término
    'no_above': 0.5,               # proporción máxima de fragmentos en los que puede aparecer
}

## **1. Carga de los fragmentos**

In [ ]:
fragments_df = pd.read_json(FRAGMENTS_PATH, lines=True)

assert len(fragments_df) > 0, 'La colección de entrada está vacía'
assert fragments_df.chunk_id.is_unique, 'Hay chunk_id duplicados'
assert fragments_df[['chunk_id', 'doc_id', 'text']].notna().all().all()
assert all(isinstance(v, str) for col in ['chunk_id', 'doc_id', 'text'] for v in fragments_df[col])
print(f'{len(fragments_df):,} fragmentos de {fragments_df.doc_id.nunique():,} documentos')
print(fragments_df.groupby('source').agg(fragmentos=('chunk_id', 'size'), documentos=('doc_id', 'nunique')))
fragments_df[['chunk_id', 'doc_id', 'title', 'source', 'word_count', 'text']].head()


## **2. Preprocesado paso a paso sobre un fragmento**

Como en la sección 1.2 de *Text Vectorization I*, aplicamos primero las cuatro etapas típicas a un único fragmento para ver qué hace cada una:

1. *Text wrangling* (limpieza del texto bruto)
2. Tokenización
3. Homogeneización (minúsculas, eliminar no alfanuméricos, lematización)
4. Limpieza (stopwords)

Elegimos un fragmento de los *Apollo press kits*, porque es donde está el problema de limpieza más importante del corpus.

In [ ]:
# Compartimos este patrón con la limpieza y las estadísticas de OCR.
SPACED_LETTERS_RE = re.compile(r'\b(?:[A-Za-z] ){2,}[A-Za-z]\b')
ocr_mask = fragments_df.text.map(lambda text: bool(SPACED_LETTERS_RE.search(text)))
preferred = fragments_df[(fragments_df.chunk_id == 'doc_3221__0020') & ocr_mask]
if not preferred.empty:
    example = preferred.iloc[0]
else:
    candidates = fragments_df[ocr_mask & fragments_df.title.str.contains('press kit', case=False, na=False)]
    if candidates.empty:
        candidates = fragments_df[ocr_mask]
    example = (candidates if not candidates.empty else fragments_df).iloc[0]
    print('Ejemplo alternativo:', example.chunk_id,
          '(el preferido no existe o no presenta el patrón de letras separadas)')
if not SPACED_LETTERS_RE.search(example.text):
    print('No se ha encontrado un ejemplo de letras separadas; revisar la muestra al ejecutar.')
print(example.chunk_id, '|', example.title, '|', example.source)
print(example.text[:700])


### *2.1. Text wrangling* 🆕 (adaptado al corpus)

En el notebook del profesor, `wrangle_text` hacía tres cosas: quitar etiquetas HTML con BeautifulSoup, quitar URLs con `re.sub` y expandir contracciones con `contractions.fix`. Las hemos revisado contra nuestros datos:

| Paso del notebook | ¿Lo aplicamos? | Motivo |
|---|---|---|
| Quitar HTML (BeautifulSoup) | **No** | El corpus ya es texto plano. Las coincidencias de `<...>` pueden ser restos de OCR que encierran texto real; BeautifulSoup podría borrarlo. Se cuentan sobre la entrada actual |
| Quitar URLs (`re.sub`) | **Sí** | El corpus ya viene sin URLs, pero es barato y las **consultas** sí podrían contenerlas |
| Expandir contracciones | **No** | spaCy ya separa `don't` → `do` + `n't`, y todas las formas expandidas (*is, are, will, not*...) son stopwords o verbos auxiliares que se eliminan después. No cambia el resultado y ahorra una dependencia |

Y añadimos estas limpiezas nuevas que necesita **nuestro** corpus (sobre todo los PDF históricos con OCR):

* **Letras separadas por espacios**: el OCR de los *Apollo press kits* produce cosas como `s t a i n l e s s steel` o `b u t i s determined`; algunos fragmentos están espaciados casi por completo. Si no hacemos nada, todas esas letras sueltas se pierden. Detectamos las secuencias de 3 o más letras sueltas, quitamos los espacios y volvemos a separar las palabras con [`wordninja`](https://github.com/keredson/wordninja), que segmenta texto sin espacios usando frecuencias de palabras del inglés: `b u t i s` → `butis` → `but is`. No es perfecto, pero los errores que deja suelen ser palabras raras que luego elimina el filtro por frecuencia documental.
* **Guiones de fin de línea**: `propul-\nsion` → `propulsion` (PDF).
* **Identificadores con guion o barra**: el tokenizador de spaCy en inglés **no separa** nombres como `STS-129`, `USA-180`, `Progress M-13` o `Soyuz MS-04`: los deja como un único token no alfabético, así que se perderían justo los nombres de las misiones. Separamos los guiones y barras entre letras/dígitos: `STS-129` → `STS 129` (después el detector de n-gramas puede volver a unirlos como `sts_129`).

Los recuentos de OCR se calculan en la celda siguiente sobre la colección cargada, sin reutilizar las cifras de la segmentación antigua. El ejemplo preferido es `doc_3221__0020`; si ya no existe o no muestra letras separadas, se busca otro caso real, dando prioridad a los press kits. Su representatividad visual se revisará al ejecutar la nueva colección.

In [ ]:
URL_RE = re.compile(r'https?://\S+|www\.\S+')
HYPHEN_BREAK_RE = re.compile(r'(\w)-[ \t]*\n\s*(\w)')
ID_SEPARATOR_RE = re.compile(r'(?<=[A-Za-z0-9])[-/](?=[A-Za-z0-9])')
WHITESPACE_RE = re.compile(r'\s+')

def fix_spaced_letters(match):
    joined = match.group(0).replace(' ', '')
    return ' '.join(wordninja.split(joined)) if CONFIG['ocr_resegment'] else joined

def wrangle_text(text):
    text = URL_RE.sub(' ', text)
    text = HYPHEN_BREAK_RE.sub(r'\1\2', text)
    text = SPACED_LETTERS_RE.sub(fix_spaced_letters, text)
    text = ID_SEPARATOR_RE.sub(' ', text)
    return WHITESPACE_RE.sub(' ', text).strip()

wrangled_example = wrangle_text(example.text)
print(wrangled_example[:600])
print('\nIdentificadores:', wrangle_text('Soyuz MS-04 docked with the ISS; STS-129 and Progress M-13 (2000/2001)'))
def compute_ocr_stats(frame):
    spaced = frame.text.map(lambda text: bool(SPACED_LETTERS_RE.search(text)))
    press_kits = frame.title.str.contains('press kit', case=False, na=False)
    return {
        'fragments_with_angle_brackets': int(frame.text.map(lambda text: bool(re.search(r'<[^>]*>', text))).sum()),
        'fragments_with_spaced_letters': int(spaced.sum()),
        'press_kit_fragments': int(press_kits.sum()),
        'press_kit_fragments_with_spaced_letters': int((spaced & press_kits).sum()),
    }

ocr_stats = compute_ocr_stats(fragments_df)
print('Recuentos de OCR en la entrada actual:', ocr_stats)


### *2.2–2.4. Tokenización, homogeneización y limpieza con spaCy*

En *Text Vectorization I* se comparaban el stemmer de NLTK, el lematizador WordNet y el de spaCy, y la conclusión era usar **spaCy**, porque lematiza usando la categoría gramatical (POS) de cada palabra en su contexto (*is* → *be*, *does* → *do*). Así que hacemos las tres etapas con spaCy directamente:

* **Tokenización**: la hace el `tokenizer` de spaCy.
* **Homogeneización**: lemas (`token.lemma_`) en minúsculas. Nos quedamos con los tokens **alfabéticos** (`is_alpha`), lo que elimina signos de puntuación, `|` de las tablas de Wikipedia, etc.
* **Filtrado por categoría gramatical**: solo las palabras con contenido (`NOUN`, `VERB`, `ADJ`, `PROPN`), como en el notebook.
* **Limpieza**: quitamos las stopwords genéricas de spaCy (`nlp.Defaults.stop_words`) más las específicas del dominio.

Como en el notebook, cargamos `en_core_web_sm` desactivando `parser` y `ner`, que no hacen falta para lematizar y ralentizan mucho el proceso. El corpus está **en inglés**, así que usamos el modelo inglés.

🆕 Dos reglas adicionales, justificadas por el dominio:

* **Longitud mínima de 2**: elimina letras sueltas (restos de OCR, abreviaturas de unidades como `m`, `s`).
* **Números e identificadores**: el notebook solo conservaba tokens alfabéticos, pero en nuestro corpus muchos nombres de misión solo se distinguen por un número o un código (*Apollo 7* vs *Apollo 8*, *Voyager 1* vs *Voyager 2*, *STS-129*, *Astra 2B*). Conservamos, en minúsculas y sin lematizar, los tokens alfanuméricos de hasta 4 caracteres que contienen algún dígito (`11`, `129`, `2b`, `1969`), sea cual sea su POS (el tagger etiqueta estos códigos de forma muy irregular: `NUM`, `NOUN`, `X`...). Los que aparecen en un solo fragmento se eliminan después con el filtro por frecuencia. Se puede desactivar con `keep_identifiers=False` para compararlo en el baseline.

In [ ]:
nlp = spacy.load(CONFIG['spacy_model'], disable=['parser', 'ner'])
print('Componentes activos:', nlp.pipe_names)

VALID_POS = set(CONFIG['valid_pos'])
STOPWORDS = set(nlp.Defaults.stop_words) | set(CONFIG['specific_stopwords'])

def normalize_token(tk):
    # Devuelve la forma normalizada del token, o None si hay que descartarlo
    text = tk.lower_
    if CONFIG['keep_identifiers'] and text.isalnum() and any(c.isdigit() for c in text):
        return text if len(text) <= CONFIG['max_identifier_len'] else None
    if not tk.is_alpha or tk.pos_ not in VALID_POS:
        return None
    lemma = tk.lemma_.lower()
    if len(lemma) < CONFIG['min_token_len'] or not lemma.isalpha():
        return None
    if lemma in STOPWORDS or tk.lower_ in STOPWORDS:
        return None
    return lemma

Veamos qué le pasa a cada token del fragmento de ejemplo:

In [ ]:
doc_example = nlp(wrangled_example)
pd.DataFrame(
    [(tk.text, tk.pos_, tk.lemma_.lower(), tk.is_stop, normalize_token(tk)) for tk in doc_example[:40]],
    columns=['token', 'POS', 'lema', 'stopword', 'resultado'],
)

In [ ]:
example_tokens = [t for t in map(normalize_token, doc_example) if t]
print(f'{len(doc_example)} tokens de spaCy -> {len(example_tokens)} tokens tras el filtrado\n')
print(example_tokens)

## **3. Preprocesado de todo el corpus**

Igual que en el ejercicio 5 de *Text Vectorization I* (`prepare_data`): aplicamos `wrangle_text`, procesamos los textos en lote con `nlp.pipe` (mucho más rápido que llamar a `nlp` texto a texto) y nos quedamos con los lemas válidos.

La misma función se usará **para las consultas**: esa es la garantía de que ambas se preprocesan igual.

Guardamos los lemas en `lemas_spacy.jsonl` para poder cambiar los n-gramas o el filtro por frecuencia sin repetir spaCy. Solo se reutilizan si los metadatos asociados verifican el orden, los identificadores y el texto de entrada, los parámetros que afectan a los lemas y las versiones del modelo y del software. Una caché antigua sin metadatos se recalcula, explicando el motivo. `FORCE_SPACY = True` permite forzar ese recálculo.

`PREPROCESSING_VERSION` es la versión explícita de las funciones de limpieza y normalización: hay que incrementarla cuando cambie su implementación. Los patrones de limpieza y las stopwords efectivas también quedan registrados. El tiempo depende de la colección y del equipo; se mide en la celda siguiente. La caché y sus metadatos se preparan en temporales y se validan mediante SHA-256.


In [ ]:
def lemmatize_texts(texts, batch_size=128):
    # Texto bruto -> lista de lemas filtrados (wrangling + spaCy + POS + stopwords)
    cleaned = (wrangle_text(t) for t in texts)
    return [[t for t in map(normalize_token, doc) if t]
            for doc in nlp.pipe(cleaned, batch_size=batch_size)]

In [ ]:
%%time
# Utilidades compartidas por la caché y el guardado final.
def fingerprint_records(records):
    digest = hashlib.sha256()
    for record in records:
        digest.update(json.dumps(record, ensure_ascii=False, sort_keys=True,
                                 separators=(',', ':'), allow_nan=False).encode('utf-8'))
        digest.update(b'\n')
    return digest.hexdigest()

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def write_json(path, data):
    with Path(path).open('w', encoding='utf-8', newline='\n') as stream:
        json.dump(data, stream, ensure_ascii=False, indent=2, allow_nan=False)
        stream.write('\n')

def write_jsonl(path, records):
    with Path(path).open('w', encoding='utf-8', newline='\n') as stream:
        for record in records:
            stream.write(json.dumps(record, ensure_ascii=False, allow_nan=False) + '\n')

def make_lemma_metadata(frame):
    assert VALID_POS == set(CONFIG['valid_pos']), 'Reejecutar la celda de spaCy tras cambiar los POS'
    assert STOPWORDS == set(nlp.Defaults.stop_words) | set(CONFIG['specific_stopwords']), 'Reejecutar la celda de spaCy tras cambiar las stopwords'
    keys = ['spacy_model', 'valid_pos', 'min_token_len', 'keep_identifiers',
            'max_identifier_len', 'ocr_resegment', 'specific_stopwords']
    return {
        'schema_version': 1,
        'implementation_version': PREPROCESSING_VERSION,
        'input': {
            'sha256': fingerprint_records(frame[['chunk_id', 'doc_id', 'text']].itertuples(index=False, name=None)),
            'n_fragments': len(frame),
            'n_documents': int(frame.doc_id.nunique()),
        },
        'parameters': {key: CONFIG[key] for key in keys},
        'spacy_version': spacy.__version__,
        'model': {'name': CONFIG['spacy_model'], 'loaded_name': nlp.meta['name'], 'version': nlp.meta['version'],
                  'language': nlp.meta['lang'], 'pipeline': list(nlp.pipe_names)},
        'wordninja_version': package_version('wordninja'),
        'valid_pos': sorted(VALID_POS),
        'stopwords': sorted(STOPWORDS),
        'cleaning_patterns': {
            name: {'pattern': pattern.pattern, 'flags': pattern.flags}
            for name, pattern in [('url', URL_RE), ('hyphen_break', HYPHEN_BREAK_RE),
                                  ('spaced_letters', SPACED_LETTERS_RE),
                                  ('id_separator', ID_SEPARATOR_RE), ('whitespace', WHITESPACE_RE)]
        },
    }

def validate_lemmas(values, frame):
    if len(values) != len(frame) or any(
        not isinstance(tokens, list) or any(not isinstance(token, str) for token in tokens)
        for tokens in values
    ):
        raise ValueError('Número o formato de las listas de lemas incompatible con la entrada')

def load_lemmas_cache(frame, expected):
    if not LEMMAS_CACHE.is_file() or not LEMMAS_META.is_file():
        return None, 'faltan la caché o sus metadatos de validación'
    try:
        metadata = json.loads(LEMMAS_META.read_text(encoding='utf-8'))
        if not isinstance(metadata, dict):
            return None, 'metadatos de caché inválidos'
        changed = [key for key, value in expected.items() if metadata.get(key) != value]
        if changed:
            return None, 'han cambiado: ' + ', '.join(changed)
        if metadata.get('cache_sha256') != sha256_file(LEMMAS_CACHE):
            return None, 'la huella de la caché no coincide (archivo alterado o guardado interrumpido)'
        with LEMMAS_CACHE.open(encoding='utf-8') as stream:
            rows = [json.loads(line) for line in stream]
        ids = [(row['chunk_id'], row['doc_id']) for row in rows]
        if ids != list(frame[['chunk_id', 'doc_id']].itertuples(index=False, name=None)):
            return None, 'los identificadores o su orden no coinciden'
        values = [row['lemmas'] for row in rows]
        validate_lemmas(values, frame)
        return values, 'caché compatible'
    except (OSError, ValueError, TypeError, KeyError) as error:
        return None, f'caché ilegible o inválida: {error}'

def save_lemmas_cache(frame, values, metadata):
    validate_lemmas(values, frame)
    with TemporaryDirectory(prefix='.lemas-', dir=OUT_DIR) as temporary:
        work = Path(temporary)
        data_path, meta_path = work / LEMMAS_CACHE.name, work / LEMMAS_META.name
        write_jsonl(data_path, (
            {'chunk_id': chunk_id, 'doc_id': doc_id, 'lemmas': tokens}
            for (chunk_id, doc_id), tokens in zip(
                frame[['chunk_id', 'doc_id']].itertuples(index=False, name=None), values, strict=True)
        ))
        write_json(meta_path, {**metadata, 'cache_sha256': sha256_file(data_path)})
        # Cada reemplazo publica un archivo completo. Si se interrumpe entre ambos,
        # la huella impide aceptar el par de archivos de ejecuciones distintas.
        os.replace(data_path, LEMMAS_CACHE)
        os.replace(meta_path, LEMMAS_META)

lemma_metadata = make_lemma_metadata(fragments_df)
lemmas, cache_reason = (None, 'FORCE_SPACY está activado') if FORCE_SPACY else load_lemmas_cache(fragments_df, lemma_metadata)
if lemmas is None:
    print('Se recalculan los lemas:', cache_reason)
    lemmas = lemmatize_texts(fragments_df.text.tolist())
    save_lemmas_cache(fragments_df, lemmas, lemma_metadata)
    print('Lemas calculados y guardados en', LEMMAS_CACHE.name)
else:
    print('Lemas cargados:', cache_reason)

print(' '.join(lemmas[0][:60]))


## **4. Detección de n-gramas**

Como en la sección 2.2 de *Text Vectorization I*, usamos `Phrases` de Gensim para detectar términos que aparecen juntos con mucha frecuencia (`service_module`, `apollo_11`...). Al haber quitado ya stopwords, se puede usar directamente. **Una sola pasada aprende bigramas** sobre los tokens de entrada: puede producir `hubble_space` o `space_telescope`, pero no garantiza el trigrama `hubble_space_telescope`. No añadimos una segunda pasada.

* `min_count = 5`: el notebook usaba 2 con 10.000 reseñas cortas; mantenemos 5 como decisión inicial del equipo para evitar asociaciones puntuales. Su efecto se mide sobre la colección cargada.
* `threshold = 20`: el mismo valor que en el notebook.

🆕 Congelamos el modelo (`freeze()`): ocupa mucho menos, es más rápido, y basta para transformar textos nuevos (las consultas), que es lo único que necesitamos de él después. Igual que dice el notebook, *el modelo de n-gramas es parte del preprocesado: cualquier texto nuevo (p. ej. una consulta) tiene que transformarse con el mismo modelo*.

⚠️ Al unir `apollo 11` en `apollo_11`, el fragmento deja de contener el unigrama `apollo`. Esto puede ayudar (términos más específicos) o perjudicar (una consulta que solo dice "Apollo" ya no encaja). Por eso es un parámetro (`use_ngrams`) que conviene comparar en el issue de los baselines.


In [ ]:
%%time
ngram_context = {
    'lemmas_sha256': fingerprint_records(lemmas),
    'config': {key: CONFIG[key] for key in ['use_ngrams', 'ngram_min_count', 'ngram_threshold']},
}
if CONFIG['use_ngrams']:
    phrase_model = Phrases(lemmas, min_count=CONFIG['ngram_min_count'], threshold=CONFIG['ngram_threshold']).freeze()
    corpus = [phrase_model[doc] for doc in lemmas]
else:
    phrase_model = None
    corpus = lemmas

ngram_counts = Counter(t for doc in corpus for t in doc if '_' in t)
print(f'{len(phrase_model.phrasegrams) if phrase_model else 0:,} n-gramas detectados; los más frecuentes en el corpus:')
print(ngram_counts.most_common(40))

In [ ]:
# Algunos n-gramas relacionados con nombres de misiones
print([g for g, _ in ngram_counts.most_common() if g.split('_')[0] in {'apollo', 'voyager', 'soyuz', 'sts', 'hubble', 'pioneer', 'gemini', 'cassini'}][:40])

## **5. Filtrado por frecuencia documental**

### *5.1. Análisis de la frecuencia documental*

Como en el ejercicio 8 de *Text Vectorization I*: construimos el diccionario de Gensim y analizamos en cuántos **fragmentos** aparece cada término (`D.dfs`).

In [ ]:
D_full = Dictionary(corpus)
n_fragments = D_full.num_docs

df_tokens = pd.DataFrame({'token': [D_full[i] for i in D_full.dfs],
                          'ndocs': list(D_full.dfs.values())}).sort_values('ndocs', ascending=False, ignore_index=True)
df_tokens['prop'] = df_tokens.ndocs / n_fragments

print(f'Vocabulario completo: {len(D_full):,} términos en {n_fragments:,} fragmentos')
print(f'Términos que aparecen en un solo fragmento: {(df_tokens.ndocs == 1).sum():,} ({(df_tokens.ndocs == 1).mean():.1%})')
print('\nMás comunes:')
print(df_tokens.head(25).to_string(index=False))

In [ ]:
print('Ejemplos de términos que solo aparecen en un fragmento (mucho ruido de OCR):')
print(df_tokens[df_tokens.ndocs == 1].sample(40, random_state=0).token.tolist())

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].hist(df_tokens.ndocs, bins=np.logspace(0, np.log10(n_fragments), 50))
ax[0].set(xscale='log', yscale='log', xlabel='Nº de fragmentos en los que aparece el término', ylabel='Nº de términos',
          title='Distribución de la frecuencia documental')
ax[1].plot(np.arange(1, len(df_tokens) + 1), df_tokens.ndocs)
ax[1].set(xscale='log', yscale='log', xlabel='Rango del término', ylabel='Nº de fragmentos', title='Frecuencia documental vs. rango (Zipf)')
plt.tight_layout()

### *5.2. Elección de `no_below` y `no_above`* 🆕

El notebook usaba `no_below=4` y `no_above=0.8` para IMDB. En nuestro caso hay que tener cuidado con `no_below`: las consultas de evaluación son de tipo *known-item* (buscan un documento concreto), y el término más útil para encontrarlo suele ser un **nombre propio poco frecuente** (el nombre del satélite o de la misión). Muchos documentos de Wikipedia tienen **un solo fragmento**, así que su nombre puede aparecer en muy pocos fragmentos del corpus.

Para elegir el umbral medimos, para varios valores, qué fracción de los términos de los **títulos** de los documentos sobrevive al filtro (los títulos se preprocesan con el mismo pipeline).

In [ ]:
titles = fragments_df.drop_duplicates('doc_id').set_index('doc_id').title
title_tokens = lemmatize_texts(titles.tolist())
if phrase_model is not None:
    title_tokens = [phrase_model[t] for t in title_tokens]

dfs_by_token = dict(zip(df_tokens.token, df_tokens.ndocs))
rows = []
for no_below in [1, 2, 3, 4, 5, 10]:
    for no_above in [0.5, 0.8]:
        keep = lambda t: no_below <= dfs_by_token.get(t, 0) <= no_above * n_fragments
        kept = [[t for t in toks if keep(t)] for toks in title_tokens]
        rows.append({'no_below': no_below, 'no_above': no_above,
                     'vocabulario': sum(keep(t) for t in dfs_by_token),
                     '% términos de títulos conservados': 100 * sum(map(len, kept)) / sum(map(len, title_tokens)),
                     '% títulos que pierden todos sus términos': 100 * np.mean([len(k) == 0 for k, t in zip(kept, title_tokens) if t])})
print(f'Títulos sin ningún término ya antes del filtro: {sum(len(t) == 0 for t in title_tokens)} de {len(title_tokens)}')
pd.DataFrame(rows).round(1)

In [ ]:
print('Términos con frecuencia documental > 50 % (los que eliminaría no_above=0.5):')
print(df_tokens[df_tokens.prop > 0.5].to_string(index=False) if (df_tokens.prop > 0.5).any() else '  ninguno')
print('\nTérminos entre el 30 % y el 50 %:')
print(df_tokens[(df_tokens.prop > 0.3) & (df_tokens.prop <= 0.5)].token.tolist())

**Decisión del equipo, pendiente de contrastar sobre la nueva colección:**

* `no_below = 2`: elimina términos que aparecen en un único fragmento. Se mantiene como compromiso inicial entre reducir ruido y conservar nombres poco frecuentes. La tabla anterior recalcula la reducción de vocabulario y la pérdida de términos de los títulos para cada umbral; no trasladamos los porcentajes de la colección antigua.
* `no_above = 0.5`: elimina términos presentes en más de la mitad de los fragmentos. La celda anterior muestra cuáles son en la entrada actual; no presuponemos que solo se elimine `mission`.

La frecuencia se mide sobre **fragmentos**, incluidos sus solapamientos: dos apariciones en fragmentos distintos pueden proceder del mismo documento original. Ambos valores se mantienen como parámetros y se revisarán con los resultados y la evaluación.


In [ ]:
dictionary_context = {
    'corpus_sha256': fingerprint_records(corpus),
    'no_below': CONFIG['no_below'], 'no_above': CONFIG['no_above'],
}
D = Dictionary(corpus)
D.filter_extremes(no_below=CONFIG['no_below'], no_above=CONFIG['no_above'], keep_n=None)   # keep_n=None: sin límite de tamaño
print(f'Diccionario: {len(D_full):,} -> {len(D):,} términos')

### *5.3. Aplicar el filtro a los fragmentos*

`D.doc2bow` ya ignora los términos que no están en el diccionario, así que para TF-IDF no haría falta nada más. Pero **BM25** (`bm25s`) indexa directamente las listas de tokens, así que eliminamos de cada fragmento los términos filtrados. Así los dos baselines parten exactamente del mismo texto preprocesado.

In [ ]:
def filter_by_dictionary(token_lists, dictionary=None):
    vocab = (D if dictionary is None else dictionary).token2id
    return [[t for t in toks if t in vocab] for toks in token_lists]

tokens_final = filter_by_dictionary(corpus)
fragments_df['n_tokens'] = [len(t) for t in tokens_final]

print(f'Tokens por fragmento: media {fragments_df.n_tokens.mean():.1f}, mediana {fragments_df.n_tokens.median():.0f}')
print(f'Tokens totales: {sum(map(len, corpus)):,} antes del filtro -> {fragments_df.n_tokens.sum():,} después')
print(f'Fragmentos vacíos tras el preprocesado: {(fragments_df.n_tokens == 0).sum()}')
print()
print(fragments_df.groupby('source').n_tokens.describe()[['mean', '50%', 'min', 'max']].round(1))

In [ ]:
fragments_df.n_tokens.hist(bins=60)
plt.xlabel('Tokens por fragmento tras el preprocesado'); plt.ylabel('Nº de fragmentos')
plt.title(f'Media: {fragments_df.n_tokens.mean():.1f} tokens (longitudes originales variables)');


In [ ]:
print('Fragmentos con menos tokens (suelen ser tablas numéricas u OCR ilegible):')
fragments_df.nsmallest(5, 'n_tokens')[['chunk_id', 'title', 'source', 'n_tokens', 'text']].assign(text=lambda d: d.text.str[:150])

## **6. Preprocesado de las consultas**

Las consultas tienen que preprocesarse **exactamente igual** que los documentos (sección 4.1 de *Text Vectorization I*): mismo `wrangle_text`, mismo pipeline de spaCy y filtros, mismo modelo de n-gramas y mismo diccionario. Por eso `preprocess_queries` no hace nada nuevo: solo encadena las mismas funciones usadas para el corpus.

In [ ]:
def preprocess_texts(texts):
    # Pipeline completo para un texto nuevo (consulta): spaCy -> n-gramas -> filtro por diccionario
    toks = lemmatize_texts(texts)
    if phrase_model is not None:
        toks = [phrase_model[t] for t in toks]
    return filter_by_dictionary(toks)

preprocess_queries = preprocess_texts

**Comprobación de consistencia** 🆕: si aplicamos a los textos de algunos fragmentos el pipeline *de consultas*, tenemos que obtener exactamente los mismos tokens que guardamos para el corpus. Si alguien cambia una de las dos partes por separado, esta celda falla.

In [ ]:
check_idx = fragments_df.sample(200, random_state=0).index
recomputed = preprocess_queries(fragments_df.loc[check_idx, 'text'].tolist())
mismatches = [i for i, toks in zip(check_idx, recomputed, strict=True) if toks != tokens_final[i]]
assert not mismatches, f'{len(mismatches)} fragmentos no coinciden'
print('OK: el pipeline de consultas reproduce exactamente los tokens del corpus en 200 fragmentos')


Ejemplos de consultas como las que generará el LLM (en inglés, el idioma del corpus). Las palabras que no están en el diccionario se descartan, igual que en los documentos.

In [ ]:
example_queries = [
    "Which satellite provided C-band television coverage over North America from a geostationary slot west of Hawaii?",
    "What food and sleep schedule did the crew follow during the first crewed Apollo flight?",
    "How did the team react when they discovered that the asteroid was ejecting particles?",
    "Who launched the Voyager 2 probe and when did it fly past Neptune?",
    "What happened to the mirror of the Hubble Space Telescope after launch?",
    "Spare parts delivered to the space station by shuttle flight STS-129",
]
for q, toks in zip(example_queries, preprocess_queries(example_queries)):
    print(q, '\n   ->', toks, '\n')

Si ya existe `queries.jsonl` en la raíz del repositorio (formato del enunciado: `{"qid": ..., "query": ..., "relevant_ids": [...], "source_chunk": ...}`), lo preprocesamos aquí. Su resultado se guarda **junto con los demás artefactos en la sección 7**, con la huella de las consultas de entrada. Si todavía no existe, no se generan consultas y la publicación final retira un posible archivo de consultas de otra ejecución.


In [ ]:
queries = None
queries_input_sha256 = None
queries_context = None
if QUERIES_PATH.exists():
    queries_input_sha256 = sha256_file(QUERIES_PATH)
    queries = pd.read_json(QUERIES_PATH, lines=True)
    assert {'qid', 'query'} <= set(queries.columns)
    assert queries.qid.notna().all() and queries.qid.is_unique
    assert queries['query'].map(lambda text: isinstance(text, str)).all()
    queries['tokens'] = preprocess_queries(queries['query'].tolist())
    queries_context = fingerprint_records([
        lemma_metadata, ngram_context, dictionary_context, sorted(D.token2id.items())
    ])
    print(f'{len(queries)} consultas preprocesadas. Consultas que se quedan sin ningún token:', (queries.tokens.str.len() == 0).sum())
    display(queries[['qid', 'query', 'tokens']].head(10))
else:
    print(f'No existe {QUERIES_PATH.name} todavía.')


## **7. Guardar los resultados**

Se comprueban la alineación de documentos, lemas y tokens, las entradas y parámetros de los modelos y la caché utilizada. Todos los resultados se serializan y comprueban primero en una carpeta temporal del mismo sistema de archivos.

`config_preprocesado.json` es el **manifiesto de una ejecución completa**: contiene las huellas de entrada, parámetros, versiones, estadísticas y SHA-256 de los artefactos. Se retira el manifiesto anterior justo antes de publicar los archivos completos y se publica el nuevo **al final**. Si se interrumpe la publicación, falta el manifiesto y los resultados no deben consumirse. Los consumidores deben comprobarlo con `validate_artifacts`; la existencia de un JSONL o un diccionario por sí sola no valida una ejecución.

Los archivos opcionales de n-gramas o consultas de una ejecución anterior se retiran cuando no corresponden a la actual. Esta celda solo se ejecutará en la segunda fase, tras regenerar la entrada.


In [ ]:
def validate_artifacts(directory):
    directory = Path(directory)
    manifest = json.loads((directory / 'config_preprocesado.json').read_text(encoding='utf-8'))
    if manifest.get('format_version') != 1 or manifest.get('complete') is not True:
        raise ValueError('Falta un manifiesto de ejecución completa compatible')
    expected = {'lemas_spacy.jsonl', 'lemas_spacy.meta.json',
                'fragmentos_preprocesados.jsonl', 'diccionario.dict'}
    if manifest['config']['use_ngrams']:
        expected.add('ngramas.phrases')
    if manifest['queries_input_sha256'] is not None:
        expected.add('consultas_preprocesadas.jsonl')
    if set(manifest['artifacts']) != expected:
        raise ValueError('La lista de artefactos no corresponde a la configuración')
    for name, digest in manifest['artifacts'].items():
        if sha256_file(directory / name) != digest:
            raise ValueError(f'Artefacto incompatible o incompleto: {name}')
    return manifest

def save_preprocessed_outputs():
    current_metadata = make_lemma_metadata(fragments_df)
    assert lemma_metadata == current_metadata, 'La entrada o el preprocesado ha cambiado: repetir desde spaCy'
    cached, reason = load_lemmas_cache(fragments_df, current_metadata)
    assert cached is not None and cached == lemmas, f'La caché no coincide con los lemas actuales: {reason}'
    assert len(lemmas) == len(corpus) == len(tokens_final) == len(fragments_df) == n_fragments
    assert D_full.num_docs == D.num_docs == n_fragments
    assert ngram_context == {
        'lemmas_sha256': fingerprint_records(lemmas),
        'config': {key: CONFIG[key] for key in ['use_ngrams', 'ngram_min_count', 'ngram_threshold']},
    }, 'Repetir la celda de n-gramas con los lemas y parámetros actuales'
    assert dictionary_context == {
        'corpus_sha256': fingerprint_records(corpus),
        'no_below': CONFIG['no_below'], 'no_above': CONFIG['no_above'],
    }, 'Repetir el filtrado documental con el corpus y parámetros actuales'
    assert (phrase_model is not None) == CONFIG['use_ngrams']
    expected_corpus = [phrase_model[tokens] for tokens in lemmas] if phrase_model is not None else lemmas
    assert corpus == expected_corpus, 'Los n-gramas no corresponden a los lemas actuales'
    assert tokens_final == filter_by_dictionary(corpus), 'Los tokens no corresponden al diccionario actual'
    assert fragments_df.n_tokens.tolist() == list(map(len, tokens_final))
    assert QUERIES_PATH.exists() == (queries is not None), 'Ha cambiado la entrada de consultas: repetir su celda'
    if queries is not None:
        assert sha256_file(QUERIES_PATH) == queries_input_sha256, 'Han cambiado las consultas de entrada'
        assert queries_context == fingerprint_records([
            lemma_metadata, ngram_context, dictionary_context, sorted(D.token2id.items())
        ]), 'Repetir las consultas con el pipeline actual'

    stats = {
        'n_fragments': int(n_fragments), 'n_documents': int(fragments_df.doc_id.nunique()),
        'vocab_before_filter': len(D_full), 'vocab_after_filter': len(D),
        'n_ngrams': len(phrase_model.phrasegrams) if phrase_model is not None else 0,
        'mean_tokens_per_fragment': round(float(fragments_df.n_tokens.mean()), 1),
        'median_tokens_per_fragment': float(fragments_df.n_tokens.median()),
        'empty_fragments_after_preprocessing': int((fragments_df.n_tokens == 0).sum()),
        'ocr': compute_ocr_stats(fragments_df), 'spacy_version': spacy.__version__,
        'gensim_version': package_version('gensim'),
    }
    with TemporaryDirectory(prefix='.resultados-', dir=OUT_DIR) as temporary:
        work = Path(temporary)
        name = 'fragmentos_preprocesados.jsonl'
        records = [
            {'chunk_id': chunk_id, 'doc_id': doc_id, 'tokens': tokens}
            for (chunk_id, doc_id), tokens in zip(
                fragments_df[['chunk_id', 'doc_id']].itertuples(index=False, name=None),
                tokens_final, strict=True)
        ]
        write_jsonl(work / name, records)
        with (work / name).open(encoding='utf-8') as stream:
            assert [json.loads(line) for line in stream] == records
        D.save(str(work / 'diccionario.dict'), separately=[])
        restored = Dictionary.load(str(work / 'diccionario.dict'))
        assert (restored.token2id, restored.dfs, restored.num_docs) == (D.token2id, D.dfs, D.num_docs)
        names = [name, 'diccionario.dict']
        if phrase_model is not None:
            phrase_model.save(str(work / 'ngramas.phrases'), separately=[])
            restored_phrases = FrozenPhrases.load(str(work / 'ngramas.phrases'))
            assert restored_phrases.phrasegrams == phrase_model.phrasegrams
            names.append('ngramas.phrases')
        if queries is not None:
            query_records = json.loads(queries.to_json(orient='records', force_ascii=False))
            assert len(query_records) == len(queries)
            write_jsonl(work / 'consultas_preprocesadas.jsonl', query_records)
            with (work / 'consultas_preprocesadas.jsonl').open(encoding='utf-8') as stream:
                assert [json.loads(line) for line in stream] == query_records
            names.append('consultas_preprocesadas.jsonl')
        manifest = {
            'format_version': 1, 'complete': True, 'config': CONFIG, 'stats': stats,
            'lemma_metadata': current_metadata, 'ngram_context': ngram_context,
            'dictionary_context': dictionary_context, 'queries_input_sha256': queries_input_sha256,
            'queries_context': queries_context,
            'artifacts': {name: sha256_file(work / name) for name in names},
        }
        manifest['artifacts'].update({
            path.name: sha256_file(path) for path in [LEMMAS_CACHE, LEMMAS_META]
        })
        write_json(work / 'config_preprocesado.json', manifest)
        # El manifiesto es el punto de confirmación del conjunto, no cada archivo.
        (OUT_DIR / 'config_preprocesado.json').unlink(missing_ok=True)
        for optional in ['ngramas.phrases', 'consultas_preprocesadas.jsonl']:
            if optional not in names:
                (OUT_DIR / optional).unlink(missing_ok=True)
        for name in names:
            os.replace(work / name, OUT_DIR / name)
        os.replace(work / 'config_preprocesado.json', OUT_DIR / 'config_preprocesado.json')
    validate_artifacts(OUT_DIR)
    return stats, manifest

stats, artifact_manifest = save_preprocessed_outputs()
for name in sorted([*artifact_manifest['artifacts'], 'config_preprocesado.json']):
    path = OUT_DIR / name
    print(f'{path.name:35s} {path.stat().st_size / 1e6:6.1f} MB')
print(json.dumps(stats, indent=2))


### *Cómo usarlo en el issue de los baselines (TF-IDF y BM25)*

```python
import json
import pandas as pd
import hashlib
from pathlib import Path
from gensim.corpora import Dictionary
from gensim.models.phrases import FrozenPhrases

# Comprobar el manifiesto antes de consumir los resultados.
out = Path('preprocesado')
manifest = json.loads((out / 'config_preprocesado.json').read_text(encoding='utf-8'))
assert manifest.get('format_version') == 1 and manifest.get('complete') is True
for name, digest in manifest['artifacts'].items():
    assert hashlib.sha256((out / name).read_bytes()).hexdigest() == digest, name
frags = pd.read_json(out / 'fragmentos_preprocesados.jsonl', lines=True)
corpus = frags.tokens.tolist()                          # para bm25s.BM25().index(corpus)
D = Dictionary.load('preprocesado/diccionario.dict')
corpus_bow = [D.doc2bow(doc) for doc in corpus]         # para TfidfModel(corpus_bow)
```

Para las consultas hay que usar la función `preprocess_queries` de este notebook (o moverla a un módulo `.py` compartido, cargando `ngramas.phrases` con `FrozenPhrases.load` y el diccionario).

---
## **Resumen de decisiones (para `MILESTONE_1.md`)**

| Paso | Decisión | Origen |
|---|---|---|
| Unidad de indexación | El fragmento es la unidad de indexación y de frecuencia documental | Enunciado del Hito 1 |
| Segmentación | Objetivo de 300 palabras; originales de hasta 400 completos; fragmentos largos de 200–400; solapamiento de 45 (15 % del objetivo); reajuste final sin pérdida de contenido | Estrategia del equipo tras revisar las indicaciones del profesor; la regla concreta de reajuste es de implementación |
| Limpieza del texto | Quitar URLs; re-segmentar con `wordninja` el OCR con letras separadas; unir palabras cortadas a final de línea; separar identificadores con guion (`STS-129` → `STS 129`). Sin BeautifulSoup ni `contractions` | TV-I §1.2.1, adaptado 🆕 |
| Tokenización y lematización | spaCy `en_core_web_sm` (sin `parser` ni `ner`), `nlp.pipe` en lotes, lemas en minúsculas | TV-I §1.2.3 y §1.3 |
| Filtrado POS | `NOUN`, `VERB`, `ADJ`, `PROPN` | TV-I §1.3 |
| Stopwords | spaCy + específicas del dominio (`et`, `al`, `fig`, `figure`, `table`...) | TV-I §1.2.4 / §1.3 |
| Otros filtros | Longitud mínima 2; se conservan números y códigos alfanuméricos de ≤ 4 caracteres (`11`, `2b`, `1969`) | 🆕 |
| N-gramas | Una pasada de bigramas con Gensim `Phrases(min_count=5, threshold=20)`, congelado | TV-I §2.2 |
| Frecuencia documental | `filter_extremes(no_below=2, no_above=0.5)` sobre fragmentos | TV-I §2.3 / Ej. 8, umbrales 🆕 |
| Consultas | Mismo pipeline (`preprocess_queries`), comprobado automáticamente | TV-I §4.1 |

El enunciado exige el preprocesado y la coherencia entre documentos y consultas. Los detalles de limpieza, identificadores, umbrales y reajuste de fragmentos son decisiones del equipo, no resultados cuantitativos del profesor. La tabla de umbrales, los gráficos y los ejemplos se validarán con la colección nueva. No se han implementado todavía los baselines ni un módulo compartido.